In [1]:
# !pip install git+https://github.com/openai/CLIP.git
# !pip install transformers
import torch
import clip
from transformers import AutoTokenizer, AutoModel
import numpy as np
from PIL import Image
import pandas as pd
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

/mnt/Work/Environments/Ubuntu/Conda/envs/cuet/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 1. CLIP model (base size, ~334M params)
# Note: This might download the model if not cached
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

# 2. BanglaBERT for richer text embeddings
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
bert_model = AutoModel.from_pretrained("csebuetnlp/banglabert").to(device)

Using device: cuda


2025-12-05 00:56:37.350598: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 00:56:37.384996: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-05 00:56:38.598033: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 00:56:38.598033: I tensorflow/core/util/port.cc:153] oneDNN custom operations ar

In [3]:
def extract_enhanced_features(image_path, raw_text, political_keyword_exists, text_based_prob):
    """
    Extracts 4 types of features:
    - CLIP image embedding (512-dim)
    - CLIP text embedding (512-dim) 
    - BanglaBERT CLS embedding (768-dim, reduced to 32-dim via PCA/AvgPool)
    - Your existing text features (2-dim)
    """
    try:
        # Load and preprocess image
        image = Image.open(image_path).convert("RGB")
        clip_image = clip_preprocess(image).unsqueeze(0).to(device)
        
        # CLIP features
        with torch.no_grad():
            # Image features
            clip_image_feat = clip_model.encode_image(clip_image)  # [1, 512]
            
            # Text features (raw text from meme)
            # CLIP context length is 77. Truncate text.
            clip_text_tokens = clip.tokenize([str(raw_text)[:77]], truncate=True).to(device)
            clip_text_feat = clip_model.encode_text(clip_text_tokens)  # [1, 512]
        
        # BanglaBERT features (for deeper text semantics)
        bert_tokens = tokenizer(
            str(raw_text),
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            bert_output = bert_model(**bert_tokens)
            bert_cls = bert_output.last_hidden_state[:, 0, :]  # [1, 768]
        
        # Reduce BERT dimensionality
        # Use average pooling to 32-dim as requested
        bert_reduced = torch.nn.functional.avg_pool1d(
            bert_cls.unsqueeze(0), kernel_size=24
        ).squeeze().cpu().numpy()  # [32]
        
        # Combine all features
        combined_feat = np.hstack([
            clip_image_feat.cpu().numpy().squeeze(),      # 512-dim
            clip_text_feat.cpu().numpy().squeeze(),       # 512-dim
            bert_reduced,                                 # 32-dim
            [political_keyword_exists, text_based_prob]   # 2-dim
        ])  # Total: 1058-dim
        
        return combined_feat
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return np.zeros(1058)

In [4]:
train_df = pd.read_csv('Dataset/Processed-csv/Processed_Train_with_bert_text_probs.csv')
test_df = pd.read_csv('Dataset/Processed-csv/Processed_Test_with_bert_text_probs.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2860, 8)
Test shape: (330, 7)


In [5]:
image_dir_train = "Dataset/Train/Image"

features = []
labels = []

print("Extracting features for training data...")
for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    img_path = os.path.join(image_dir_train, row["Image_name"])
    
    feat = extract_enhanced_features(
        img_path,
        row["Text"],
        row["political_keyword_exists"],
        row["text_based_prob"]
    )
    
    features.append(feat)
    labels.append(row["Label"])

X = np.vstack(features)
labels = np.array(labels)

# Encode Labels
le = LabelEncoder()
y = le.fit_transform(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", le.classes_)

Extracting features for training data...


100%|██████████| 2860/2860 [01:09<00:00, 41.08it/s]

X shape: (2860, 1058)
y shape: (2860,)
Classes: ['NonPolitical' 'Political']


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (important for CLIP features)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [7]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,  # Slightly shallower for high-dimensional features
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    early_stopping_rounds=50
)

print("Training XGBoost...")
xgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50
)

Training XGBoost...
[0]	validation_0-logloss:0.66919	validation_1-logloss:0.67632
[0]	validation_0-logloss:0.66919	validation_1-logloss:0.67632
[50]	validation_0-logloss:0.15207	validation_1-logloss:0.26980
[50]	validation_0-logloss:0.15207	validation_1-logloss:0.26980
[100]	validation_0-logloss:0.05847	validation_1-logloss:0.21000
[100]	validation_0-logloss:0.05847	validation_1-logloss:0.21000
[150]	validation_0-logloss:0.02707	validation_1-logloss:0.19276
[150]	validation_0-logloss:0.02707	validation_1-logloss:0.19276
[200]	validation_0-logloss:0.01516	validation_1-logloss:0.18623
[200]	validation_0-logloss:0.01516	validation_1-logloss:0.18623
[250]	validation_0-logloss:0.00971	validation_1-logloss:0.18525
[250]	validation_0-logloss:0.00971	validation_1-logloss:0.18525
[262]	validation_0-logloss:0.00888	validation_1-logloss:0.18597
[262]	validation_0-logloss:0.00888	validation_1-logloss:0.18597


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,'logloss'


In [8]:
y_val_pred = xgb.predict(X_val)
y_val_prob = xgb.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.3f}")
print(f"F1-score: {f1_score(y_val, y_val_pred, average='macro'):.3f}")
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

Accuracy: 0.930
F1-score: 0.915
              precision    recall  f1-score   support

NonPolitical       0.94      0.97      0.95       401
   Political       0.91      0.85      0.88       171

    accuracy                           0.93       572
   macro avg       0.92      0.91      0.91       572
weighted avg       0.93      0.93      0.93       572



In [9]:
image_dir_test = "Dataset/Test/Image"

test_features = []

print("Extracting features for test data...")
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img_path = os.path.join(image_dir_test, row["Image_name"])
    
    feat = extract_enhanced_features(
        img_path,
        row["Text"],
        row["political_keyword_exists"],
        row["text_based_prob"]
    )
    
    test_features.append(feat)

X_test = np.vstack(test_features)
X_test = scaler.transform(X_test)

print("X_test shape:", X_test.shape)

Extracting features for test data...


100%|██████████| 330/330 [00:09<00:00, 35.16it/s]

X_test shape: (330, 1058)


In [10]:
test_pred_prob = xgb.predict_proba(X_test)[:, 1]
test_predictions = (test_pred_prob >= 0.5).astype(int)
test_labels = le.inverse_transform(test_predictions)

submission_df = pd.DataFrame({
    'Image_name': test_df['Image_name'],
    'Label': test_labels
})

submission_file_path = 'submission_clip_banglabert_xgboost.csv'
submission_df.to_csv(submission_file_path, index=False)

print(f"Submission file saved to {submission_file_path}")
submission_df.head()

Submission file saved to submission_clip_banglabert_xgboost.csv


,Image_name,Label
0,test0001.jpg,Political
1,test0002.jpg,NonPolitical
2,test0003.jpg,NonPolitical
3,test0004.jpg,Political
4,test0005.jpg,NonPolitical
